In [41]:
import pandas as pd
import numpy as np
import re
import nltk
import pickle

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import classification_report, accuracy_score

In [42]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Mathavan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Mathavan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [43]:
df = pd.read_csv("civicdex_adapted.csv")

In [44]:
df = df[['english_gloss', 'department']]

In [45]:
lemmatizer = WordNetLemmatizer()

stop_words = set(stopwords.words('english'))

def clean_text(text):

    text = text.lower()

    text = re.sub(r'http\S+|[^a-zA-Z ]', '', text)

    words = text.split()

    words = [word for word in words if word not in stop_words]

    words = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(words)

In [46]:
df['cleaned_text'] = df['english_gloss'].apply(clean_text)

In [47]:
print(df[['english_gloss', 'cleaned_text']].head())

                                       english_gloss  \
0                   Garbage not collected for 3 days   
1  What is the status of my ration card application?   
2             How to apply for new water connection?   
3                        Street light is not working   
4                        Drain is completely blocked   

                     cleaned_text  
0           garbage collected day  
1  status ration card application  
2      apply new water connection  
3            street light working  
4        drain completely blocked  


In [48]:
tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(df['cleaned_text'])

In [49]:
y = df['department']

In [50]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [51]:
model = LinearSVC()

model.fit(X_train, y_train)

LinearSVC()

In [52]:
predictions = model.predict(X_test)

In [53]:
print("Accuracy:", accuracy_score(y_test, predictions))

print(classification_report(y_test, predictions))

Accuracy: 0.9242424242424242
                          precision    recall  f1-score   support

     Drainage Department       1.00      1.00      1.00        20
  Electricity Department       0.88      1.00      0.94        22
Public Health Department       1.00      0.67      0.80        12
      Revenue Department       1.00      1.00      1.00        18
        Roads Department       0.89      0.89      0.89        18
   Sanitation Department       0.95      0.90      0.92        20
    Transport Department       0.67      0.80      0.73         5
 Water Supply Department       0.93      1.00      0.97        14
      Welfare Department       0.67      0.67      0.67         3

                accuracy                           0.92       132
               macro avg       0.89      0.88      0.88       132
            weighted avg       0.93      0.92      0.92       132



In [54]:
sample = ["There is no water supply in my area"]

sample_clean = [clean_text(text) for text in sample]

sample_vector = tfidf.transform(sample_clean)

prediction = model.predict(sample_vector)

print(prediction)

['Water Supply Department']


In [55]:
pickle.dump(model, open("department_model.pkl", "wb"))

pickle.dump(tfidf, open("tfidf.pkl", "wb"))